In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import json

In [111]:
def check_mission_data(df:pd.DataFrame) -> dict:
    missing_data = df.isnull().sum()
    missing_data_pct = (missing_data / len(df)) * 100
    
    column_types = {}
    for col in df.columns:
        if df[col].dtype in ['int64', 'float64']:
            column_types[col] = 'numeric'
        else:
            column_types[col] = 'categorical'

    
    return {
        'has_missing':missing_data.sum() > 0,
        'total_missing': missing_data.sum(), 
        'missing_data': missing_data[missing_data > 0].to_dict(),
        'missing_pct':missing_data_pct.to_dict(), 
        'columns': df.columns[df.isnull().any()].tolist(),
        'column_types': column_types
        }
    

def check_outliers_top_k(df:pd.DataFrame, column:str, k:int = 10) -> dict:
    
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    upper_iqr = Q3 + 1.5 * IQR
    lower_iqr = Q1 - 1.5 * IQR
    
    high_outliers = df[df[column] > upper_iqr].nlargest(k, column)
    low_outliers = df[df[column] < lower_iqr].nsmallest(k, column)
    
    top_outliers = pd.concat([high_outliers, low_outliers])

    if len(top_outliers) > k:
        top_outliers['deviation'] = abs(top_outliers[column] - df[column].median())
        top_outliers = top_outliers.nlargest(k, 'deviation')
        top_outliers = top_outliers.drop(columns=['deviation'])
    
    if len(top_outliers) == 0:
        return {
            'has_outliers': False,
            'k_outliers': 0,
            'total_outliers': 0,
            'values_to_remove': [], 
            
        }
    
    
    values_to_remove = top_outliers[column].tolist()
    
    return {
        'has_outliers': True,
        'k_outliers': len(values_to_remove),
        'total_outliers': len(df[(df[column] > upper_iqr) | (df[column] < lower_iqr)]),
        'values_to_remove': values_to_remove,  
        'upper_threshold': float(top_outliers[column].max()),
        'lower_threshold': float(top_outliers[column].min()),
        'k': k,
        
    }

    

def check_duplicates(df:pd.DataFrame) -> dict:
    total_duplicates = df.duplicated().sum()
    return {
        'has_duplicates': total_duplicates > 0, 
        'total_duplicates':total_duplicates
        }
    

def check_positive(df:pd.DataFrame, column) -> dict:
    total_invalid = len(df[df[column] < 0])
    return {'has_negative': (total_invalid > 0), 'total_negative': total_invalid}
    


In [108]:
def make_summary(df_dict:dict) -> dict:
    summary = {}
    
    
    for name, df in df_dict.items():
        numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
        positive_cols = ['item_price', 'item_cnt_day', 'date_block_num']
        
        summary[name] = {}
            
        summary[name]['missing'] = check_mission_data(df)
        summary[name]['missing']['suggested_action'] = ""
        
        
        summary[name]['duplicates'] = check_duplicates(df)
        summary[name]['duplicates']['suggested_action'] = ""
        
        
        summary[name]['outliers'] = {}
        
        for col in numeric_cols:
            summary[name]['outliers'][col] = check_outliers_top_k(df, col, 2)
            summary[name]['outliers'][col]['suggested_action'] = ''
            
            
       
        summary[name]['negative_values'] = {}
        for col in numeric_cols:
            if (col in positive_cols) or ('price' in col or 'cnt' in col or 'value' in col):    
                summary[name]['negative_values'][col] = check_positive(df, col)
                summary[name]['negative_values'][col]['suggested_action'] = ''
        
        
        summary[name]['dtypes'] = df.dtypes.astype(str).to_dict()
       
       
        summary[name]['shape'] = {
            'rows': len(df),
            'columns': len(df.columns)
        }
    
    return summary

In [89]:
PATH = '../raw_data/'
# PATH = '../tmp/'

In [90]:
shops = pd.read_csv(PATH +'shops.csv')
items = pd.read_csv(PATH +'items.csv')
item_categories = pd.read_csv(PATH +'item_categories.csv')
sales_train = pd.read_csv(PATH + 'sales_train.csv')

In [109]:
dataframes = {
    'sales_train': sales_train,
    'shops':shops,
    'items':items,
    'item_categories':item_categories
}

issues_summary = make_summary(dataframes)

In [110]:
with open('../json/issues_summary.json', 'w') as f:
    json.dump(issues_summary, f, indent=2, default=str)